In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io.wavfile import write
import librosa 

Learning Outcome: The student will understand what a spectrogram is and the bridge between the FFT. They will clarify why spectograms are useful for analysing signals and how different window frames affect the output of the short term fourier transform.

In [ ]:
Fs = 16000
dur = 1.0
t = np.linspace(0, dur, int(Fs*dur))
window_type = 'boxcar'
#TODO create a linear chirp signal, starting at a particular frequncy and rising linearly

nperseg = 1024 #This is the number of samples
noverlap = nperseg//4 #Represents the percentage overlap

#TODO Try different window types and compute the spectogram
#TODO Plot the spectogram with the different window types


1) What difference do you notice in the sharpness and smoothness of the chirp line as you change the window? Explain how the window length and shape affect frequency resolution and time resolution.


Learning Outcomes: Use real audio such as a piano song to observe the frequencies that the signal is composed of. The student will observe and match the audio with the spectrogram which will clarify their understanding. In addition they will filter and alter the audio by using the inverse FFT and then listen to the affect of masking different frequencies. The students should be encouraged to use a piece fo music which they enjoy and recognise to observe the effects of signal filtering clearly.


In [ ]:

AUDIO_PATH = ""  # change to a .wav or .mp3 on your machine

def load_audio(path):
    try:
        y, fs = librosa.load(path, sr=None, mono=True)
        return fs, y.astype(np.float64)
    except Exception:
        fs = 16_000
        t = np.linspace(0, 2.0, int(fs*2.0), endpoint=False)
        y = signal.chirp(t, f0=200, f1=2000, t1=2.0, method="linear")
        print("Note: could not read file; using a synthetic chirp instead.")
        return fs, y

fs, y = load_audio(AUDIO_PATH)

Tmax = min(len(y)/fs, 60.0)
y = y[:int(Tmax*fs)]

# TODO Compute and create a spectogram

# TODO Plot log-spectrogram 

# TODO Invert the spectogram

# TODO pad/truncate to original length for error metric

# TODO enhance the magnitude of the original signal by 0.8 and 1.2

# TODO plot the new spectogram

# TODO use inverse istft and listen to output

# TODO plot the istft


Apply spectral gamma with 0.8 and 1.2, reconstruct audio, RMS-normalise both to the original, and compare spectrograms and short waveform snippets.
1) Which gamma compresses vs expands spectral dynamic range? Describe audible changes and relate them to changes you observe in the spectrogram.

Learning Outcomes: The aim is to provide a clear understanding of modulation and the difference between AM and FM. This practice is concise to allow a practical implementation of a carrier and message frequency.


In [ ]:

Fs = 48_000
T  = 0.05
t  = np.linspace(0, T, int(Fs*T), endpoint=False)


fm = 200.0                        # message frequency
m  = np.sin(2*np.pi*fm*t)         # message 

fc = 5000.0                      # carrier
carrier = np.cos(2*np.pi*fc*t)

#TODO compute the amplitude modulation


#TODO compute the frequency modulation


#TODO plot the amplitude modulation

#TODO plot the frequency modulation



Explain why the FM spectrum is wider even though the message frequency is the same as in AM.


Learning Outcomes: The student should record their own voice for the voiced and unvoiced phonemes and use this audio to compute the zero crossing rate. The zero crossing rate and energy of the signal will demonstrate the difference between voiced and unvoiced phonemes by the signal oscillation.

In [ ]:
AUDIO_PATH = ""  
def load_audio(path):

    try:
        y, fs = librosa.load(path, sr=None, mono=True)
        return fs, y.astype(float)
    except Exception:
        print("Could not read file; falling back to synthetic.")
        return load_audio(None)

# TODO Upload your audio of voiced vs unvoiced phonemes
fs, x = load_audio(AUDIO_PATH)


frame_ms = 25
hop_ms   = 10
Nf = int(fs*frame_ms/1000)
H  = int(fs*hop_ms/1000)
win = np.hanning(Nf)

#TODO iterate through a for loop to compute the hanns frame

#TODO Compute the energy and the zcr and plot this



Identify areas on the graph that are voiced and unvoiced phonemes and briefly explain why this is.


Learning Outcome: By the end of this section, students will be able to use an FFT to show that narrowing a wavefunction in position broadens its spectrum in k, and relate this to the uncertainty bound $\Delta$ $x$ $\,$ $\Delta$ p $\ge$ $\hbar/2$ via $p=\hbar k$.

In [ ]:
def normalize_probability_density(pdf, dx):


    area = np.sum(pdf) * dx

    return pdf / area


def mean_and_std(x, pdf, dx):
    """Compute mean and standard deviation of x under pdf using Riemann sums."""


    pdf = normalize_probability_density(pdf, dx)
    mu = np.sum(x * pdf) * dx
    var = np.sum((x - mu) ** 2 * pdf) * dx
    
    return mu, np.sqrt(var)


def k_grid(N, dx):
    """
    FFT frequency grid in angular wavenumber k [rad / unit length].
    numpy.fft.fftfreq gives cycles/unit then multiply by 2 pi to get rad/unit.
    """


    f = np.fft.fftfreq(N, d=dx)
    k = 2 * np.pi * f
    return np.fft.fftshift(k)


def fft_k_space(psi_x, dx):
    """
    Compute a k-space wavefunction from psi(x) using FFT.
    """


    psi_k = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(psi_x))) * dx / np.sqrt(2*np.pi)
    return psi_k


def make_gaussian_packet(x, sigma_x, k0=0.0):


    psi = np.exp(-(x**2) / (4 * sigma_x**2)) * np.exp(1j * k0 * x)
    return psi


def report_uncertainties(x, psi_x, hbar=1.0):
    """
    Compute delta x from |psi(x)|^2 and delta k from |psi(k)|^2, then delta p = h_bar delta k.
    Returns (dx_std, dk_std, dp_std, product_dx_dk, product_dx_dp, k, pk, px).
    """
    N = x.size
    dx = x[1] - x[0]
    k = k_grid(N, dx)
    dk = k[1] - k[0]

    # Position space probability 


    px = np.abs(psi_x)**2
    px = normalize_probability_density(px, dx)
    _, dx_std = mean_and_std(x, px, dx)

    #  k-space probability 


    psi_k = fft_k_space(psi_x, dx)
    pk = np.abs(psi_k)**2
    pk = normalize_probability_density(pk, dk)
    _, dk_std = mean_and_std(k, pk, dk)

    #  Translate to momentum


    dp_std = hbar * dk_std

    return dx_std, dk_std, dp_std, dx_std * dk_std, dx_std * dp_std, k, pk, px


def plot_x_and_k(x, px, k, pk, title):

    plt.figure()
    plt.plot(x, px)
    plt.xlabel("x")
    plt.ylabel(r"$|\psi(x)|^2$ (normalized)")
    plt.title(title + " — position probability")
    plt.tight_layout()

    plt.figure()
    plt.plot(k, pk)
    plt.xlabel("k (rad / unit length)")
    plt.ylabel(r"$|\tilde{\psi}(k)|^2$ (normalized)")
    plt.title(title + " — k-space probability")
    plt.tight_layout()



# TODO try changing N and L and see how it affects resolution in x and k.
N = 2**12          # 4096 points
L = 200.0          # domain length in x-units
dx = L / N
x = (np.arange(N) - N/2) * dx

# Choose ħ = 1 to keep units simple
hbar = 1.0

sigmas = [0.5, 2.0, 10.0]  
for sigma_x in sigmas:
    # TODO generate psi(x) for each sigma_x

    # TODO compute uncertainties in x and k and momentum

    print(f"Gaussian sigma_x={sigma_x:>4}: delta x={dx_std:.4f}, delta k={dk_std:.4f}, delta x delta k={prod_xk:.4f} ")
    plot_x_and_k(x, px, k, pk, title=f"Gaussian (sigma_x={sigma_x})")


plt.show()

1. As $\sigma_x$ decreases what happens to $\Delta k$? Is it roughly inverse to $\Delta x$?

2. For the Gaussian cases, what do you get for $\Delta x\,\Delta k$? How close is it to $1/2$?

3. Using $p=\hbar k$, compute $\Delta p$. Verify whether $\Delta x\,\Delta p \ge \hbar/2$ holds (here $\hbar=1$).

4. Explain breifly how this relates to heisenberg's uncertainty principle.